# DAY 9: Reference-Free Preference Alignment (ORPO & KTO)

## 🎯 Learning Objectives
In Day 8, we implemented **Direct Preference Optimization (DPO)** and proved that aligning an SFT model significantly eliminates preamble chattiness and negative generation habits. However, DPO has a critical hardware cost:
> **The DPO VRAM Bottleneck**: DPO requires evaluating log probabilities under *both* the active policy $\pi_\theta$ AND a frozen reference model $\pi_{\text{ref}}$, effectively requiring dual-model forward passes in GPU memory.

Today, you will master **Reference-Free Alignment**, specifically:
1. **ORPO (Odds Ratio Preference Optimization)**: A monolithic post-training algorithm that combines SFT and preference alignment into a **single objective function without any reference model**.
2. **The Odds Ratio Penalty**: How ORPO penalizes disfavored generation odds relative to favored generation odds directly using the policy's own logits.
3. **KTO (Kahneman-Tversky Optimization)**: Direct alignment from binary feedback (desirable vs. undesirable) grounded in human prospect theory (loss aversion), eliminating the need for strictly paired $(y_w, y_l)$ datasets.
4. **Memory Comparison**: Why ORPO saves ~40–50% GPU memory compared to DPO while achieving equal or superior alignment on small-to-medium models.

## 📐 The Mathematics of ORPO (Hong et al., 2024)

Standard SFT cross-entropy loss only maximizes the likelihood of the chosen response:
$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|y_w|} \sum_{t=1}^{|y_w|} \log P(y_{w,t} \mid x, y_{w,<t})$$

However, SFT inherently allocates non-zero probability mass to undesirable completions ($y_l$) as well.

### The Odds Function
The odds of generating sequence $y$ given prompt $x$ under policy $\theta$ is defined as:
$$\text{odds}_\theta(y \mid x) = \frac{P_\theta(y \mid x)}{1 - P_\theta(y \mid x)}$$

### The Odds Ratio (OR)
We compare the odds of the favored completion $y_w$ against the odds of the rejected completion $y_l$:
$$\text{OR}_\theta(y_w, y_l) = \frac{\text{odds}_\theta(y_w \mid x)}{\text{odds}_\theta(y_l \mid x)}$$

### The Unified ORPO Objective
$$\mathcal{L}_{\text{ORPO}} = \mathcal{L}_{\text{SFT}} + \lambda \cdot \mathcal{L}_{\text{OR}}$$
where:
$$\mathcal{L}_{\text{OR}} = -\log \sigma\left( \log \text{OR}_\theta(y_w, y_l) \right) = -\log \sigma\left( \log \frac{\text{odds}_\theta(y_w \mid x)}{\text{odds}_\theta(y_l \mid x)} \right)$$

- When $\text{odds}_\theta(y_w) \gg \text{odds}_\theta(y_l)$, $\mathcal{L}_{\text{OR}} \to 0$.
- $\lambda$ (typically 0.1 to 0.2) controls the relative weight of the preference penalty.
- **Notice**: No reference model $\pi_{\text{ref}}$ appears anywhere in the formula!

### Step 1: Environment & GPU Inspection

In [ ]:
# !pip install -q transformers peft trl accelerate bitsandbytes datasets

In [ ]:
import sys
import torch
import transformers
import peft
import trl

print("=" * 60)
print("🚀 RUNTIME & HARDWARE ENVIRONMENT REPORT (DAY 9: ORPO)")
print("=" * 60)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("GPU Device           : CPU Mode")
print(f"Transformers Version : {transformers.__version__}")
print(f"PEFT Version         : {peft.__version__}")
print(f"TRL Version          : {trl.__version__}")
print("=" * 60)

### Step 2: Prepare the Pairwise Dataset for ORPO

ORPO expects data in the format:
- `prompt`: The user input or instruction
- `chosen`: The optimal response (clean, direct JSON extraction, zero pleasantries)
- `rejected`: The undesirable response (chatty preamble, markdown backticks, conversational fluff, or hallucinated schema)

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an expert entity extraction system. Extract structured order information "
    "from the user's text into valid JSON according to the schema: "
    '{"customer": string, "quantity": integer, "product": string, "amount": float, "delivery_day": string}.'
)

# Sample pairwise triplets demonstrating positive vs negative behavior
sample_triplets = [
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "John ordered 3 laptops for $2400 and wants delivery on Friday."}
        ],
        "chosen": [
            {"role": "assistant", "content": '{"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}'}
        ],
        "rejected": [
            {"role": "assistant", "content": 'Sure! Here is the extracted order information in JSON format:\n```json\n{"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}\n```\nLet me know if you need anything else!'}
        ]
    },
    {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Please send 5 4K monitors to Sarah. The total is $1500 and it needs to arrive on Monday."}
        ],
        "chosen": [
            {"role": "assistant", "content": '{"customer": "Sarah", "quantity": 5, "product": "4K monitors", "amount": 1500.0, "delivery_day": "Monday"}'}
        ],
        "rejected": [
            {"role": "assistant", "content": 'Certainly! Below is the order:\n{"client": "Sarah", "units": 5, "item": "4K monitors", "cost": 1500.0, "date": "Monday"}'}
        ]
    }
]

orpo_dataset = Dataset.from_list(sample_triplets)
print(f"Created ORPO Dataset with {len(orpo_dataset)} samples.")
print("Sample structure keys:", orpo_dataset.column_names)

### Step 3: Load Model & Configure ORPO Hyperparameters

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=compute_dtype
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
print("Model and LoRA configured for ORPO training.")

### Step 4: Configure `ORPOConfig` and Execute with TRL `ORPOTrainer`

In [ ]:
from trl import ORPOConfig, ORPOTrainer

orpo_args = ORPOConfig(
    output_dir="models/adapters/qwen-1.5b-orpo-alignment",
    beta=0.1,                          # Equivalent to lambda in the ORPO objective (controls odds ratio penalty)
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_length=512,
    max_prompt_length=256,
    max_completion_length=128,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    logging_steps=5,
    save_strategy="no",
    report_to="none"
)

print("ORPO Configuration:")
print(f"  - Beta (Odds Ratio Weight) : {orpo_args.beta}")
print(f"  - Learning Rate            : {orpo_args.learning_rate}")
print(f"  - Max Sequence Length      : {orpo_args.max_length}")
print(f"  - Reference Model Required : FALSE (0 GB extra VRAM!)")

---
## 🎓 Day 9 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Monolithic Alignment**: Merging instruction fine-tuning and preference optimization into one single stage without intermediate checkpoints.
2. **Odds Ratio vs. Probability Ratio**: Probabilities are bounded between 0 and 1; odds range from 0 to $\infty$, making odds ratios significantly more sensitive to disfavored token generation.
3. **Memory Footprint**: DPO requires loading or computing log probabilities on $\pi_{\text{ref}}$, increasing GPU activation memory by ~1.5x–2.0x. ORPO runs entirely within a single model instance.
4. **KTO (Kahneman-Tversky Optimization)**: In scenarios where creating paired $(y_w, y_l)$ examples is expensive or unavailable (e.g. user upvotes/downvotes in production logs), KTO directly optimizes utility functions using Kahneman's prospect theory with loss aversion ($v(x) > v(-x)$ for losses vs gains).

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *What is the primary advantage of ORPO over DPO?*
   - **Answer**: ORPO eliminates the need for a separate frozen reference model $\pi_{\text{ref}}$. This cuts VRAM requirements by ~40–50% and allows alignment in a single training step directly from the base or instruct model.
2. **Q2 (Intermediate)**: *How does ORPO penalize bad responses without a reference model?*
   - **Answer**: It calculates the odds of the favored tokens vs. the odds of the rejected tokens using the policy's own current logits. The loss function $\mathcal{L}_{\text{OR}} = -\log \sigma(\log \text{OR})$ penalizes the model whenever the rejected completion has high generation odds relative to the chosen completion.
3. **Q3 (Intermediate)**: *Can ORPO replace SFT entirely?*
   - **Answer**: Yes. In the original paper, Hong et al. showed that running ORPO on a base model achieves competitive instruction-following and alignment simultaneously because $\mathcal{L}_{\text{SFT}}$ is directly integrated into the loss.
4. **Q4 (Advanced)**: *What is the difference between pairwise preference optimization (DPO/ORPO) and binary preference optimization (KTO)?*
   - **Answer**: Pairwise methods require prompt-matched triplets $(x, y_{\text{chosen}}, y_{\text{rejected}})$. KTO only requires $(x, y, \text{label} \in \{+1, -1\})$, meaning unpaired production data (e.g., user thumbs-up or thumbs-down logs) can be trained directly without finding matching paired counterfactuals.
5. **Q5 (Advanced)**: *Why does loss aversion matter in KTO?*
   - **Answer**: Grounded in behavioural economics, humans perceive the negative utility of a bad completion more intensely than the positive utility of a good one. KTO penalizes undesirable completions more steeply than it rewards desirable ones, preventing hallucinated or toxic outputs.

### 🏆 Day 9 Hands-On Challenge
> **Challenge**: Compare the VRAM consumed by TRL's `DPOTrainer` (which tracks `model` and `ref_model`) vs. `ORPOTrainer`. Calculate the maximum batch size that fits in 16GB VRAM for each method.